In [1]:
import duckdb
import pandas as pd
from pathlib import Path

DATA = Path("data")
con = duckdb.connect()
train = (DATA / "application_train.csv").as_posix()

summary = con.execute(f"SUMMARIZE SELECT * FROM read_csv_auto('{train}')").df()
summary["null_percentage"] = summary["null_percentage"].astype(float)

missing = summary[["column_name", "column_type", "null_percentage"]] \
    .sort_values("null_percentage", ascending=False)

print("Columns with any missing:", (missing["null_percentage"] > 0).sum(), "of", len(missing))
print("Columns over 50% missing:", (missing["null_percentage"] > 50).sum())
missing.head(15)

Columns with any missing: 64 of 122
Columns over 50% missing: 41


,column_name,column_type,null_percentage
48,COMMONAREA_AVG,DOUBLE,69.87
62,COMMONAREA_MODE,DOUBLE,69.87
76,COMMONAREA_MEDI,DOUBLE,69.87
84,NONLIVINGAPARTMENTS_MEDI,DOUBLE,69.43
70,NONLIVINGAPARTMENTS_MODE,DOUBLE,69.43
56,NONLIVINGAPARTMENTS_AVG,DOUBLE,69.43
86,FONDKAPREMONT_MODE,VARCHAR,68.39
54,LIVINGAPARTMENTS_AVG,DOUBLE,68.35
82,LIVINGAPARTMENTS_MEDI,DOUBLE,68.35
68,LIVINGAPARTMENTS_MODE,DOUBLE,68.35


In [2]:
con.execute(f"""
    SELECT
      CASE WHEN COMMONAREA_AVG IS NULL THEN 'building info missing'
           ELSE 'building info present' END AS grp,
      COUNT(*) AS people,
      ROUND(100.0 * AVG(TARGET), 2) AS default_pct
    FROM read_csv_auto('{train}')
    GROUP BY grp
""").df()

,grp,people,default_pct
0,building info present,92646,6.91
1,building info missing,214865,8.57


In [3]:
text_cols = summary.loc[summary["column_type"] == "VARCHAR", "column_name"].tolist()
print("Text columns:", len(text_cols))

for c in text_cols:
    r = con.execute(f"""
        SELECT COUNT(*) FROM read_csv_auto('{train}')
        WHERE {c} IN ('XNA', 'XAP', 'Unknown', 'unknown')
    """).fetchone()[0]
    if r > 0:
        print(f"{c:28s} {r:>7,} fake-looking values")

Text columns: 15
CODE_GENDER                        4 fake-looking values
NAME_FAMILY_STATUS                 2 fake-looking values
ORGANIZATION_TYPE             55,374 fake-looking values


In [4]:
con.execute(f"""
    SELECT
      ORGANIZATION_TYPE = 'XNA'   AS org_is_xna,
      DAYS_EMPLOYED   = 365243    AS days_is_fake,
      COUNT(*) AS people
    FROM read_csv_auto('{train}')
    WHERE ORGANIZATION_TYPE = 'XNA' OR DAYS_EMPLOYED = 365243
    GROUP BY 1, 2
""").df()

,org_is_xna,days_is_fake,people
0,True,True,55374


In [5]:
days_cols = [c for c in summary["column_name"] if c.startswith("DAYS_")]

for c in days_cols:
    r = con.execute(f"""
        SELECT COUNT(*) FILTER (WHERE {c} > 0),
               MIN({c}), MAX({c})
        FROM read_csv_auto('{train}')
    """).fetchone()
    print(f"{c:26s} positive={r[0]:>7,}   min={r[1]}   max={r[2]}")

DAYS_BIRTH                 positive=      0   min=-25229   max=-7489
DAYS_EMPLOYED              positive= 55,374   min=-17912   max=365243
DAYS_REGISTRATION          positive=      0   min=-24672.0   max=0.0
DAYS_ID_PUBLISH            positive=      0   min=-7197   max=0
DAYS_LAST_PHONE_CHANGE     positive=      0   min=-4292.0   max=0.0


## Stage 3 findings: data quality (application_train)

- **64 of 122 columns** have missing values; **41 are over 50% missing**.
- The most-missing columns describe the applicant's **building** (~70% missing). Each appears 3 times (`_AVG`, `_MODE`, `_MEDI`).
- **Missingness is informative:** building info missing → 8.57% default vs 6.91% when present.
- **Placeholder codes:** `DAYS_EMPLOYED = 365243` and `ORGANIZATION_TYPE = 'XNA'` mark the **same 55,374 applicants** (verified exact overlap), 99.96% pensioners.
- **Small placeholders:** `CODE_GENDER` has 4 `XNA`; `NAME_FAMILY_STATUS` has 2 `Unknown`.
- **All other `DAYS_` columns are valid** (no positive values).
- Plan: convert placeholders to missing, add a pensioner flag, keep missing-value flags, cap the extreme income.